# Phase 4 — Food Image-to-Recipe Retrieval · Kaggle GPU Run

**Goal:** Train and evaluate the joint embedding model (baseline concat vs. attention fusion)  
**Dataset:** [pes12017000148/food-ingredients-and-recipe-dataset-with-images](https://www.kaggle.com/datasets/pes12017000148/food-ingredients-and-recipe-dataset-with-images)  
**GPU:** T4 × 2 or P100  
**Reports:** medR + R@1/5/10 for im2recipe + recipe2im at 1k and full-gallery, mean ± std over 3 seeds

---
**Run order:**
1. Install → Write source → Configs
2. Precompute CLIP ViT-B/32 image features
3. Train baseline (concat fusion) × 3 seeds
4. Train attention fusion × 3 seeds
5. Evaluate + results table

## 1. Install dependencies

In [ ]:
!pip install -q open-clip-torch==2.26.1 transformers==4.44.2 omegaconf==2.3.0 tqdm tensorboard

## 2. Write source files

In [ ]:
import os

DIRS = [
    "src", "src/data", "src/models", "src/losses",
    "src/eval", "src/training", "src/utils",
    "configs", "data/processed",
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

# __init__ files
for pkg in ["src", "src/data", "src/models", "src/losses", "src/eval", "src/training", "src/utils"]:
    open(f"{pkg}/__init__.py", "w").close()

print("Directories ready.")

In [ ]:
# ── src/utils/config.py ──────────────────────────────────────────────────────
with open("src/utils/config.py", "w") as f:
    f.write('''
from __future__ import annotations
from pathlib import Path
from omegaconf import DictConfig, OmegaConf

_CONFIG_DIR = Path("configs")

def load_config(path: str, overrides: list[str] | None = None) -> DictConfig:
    cfg_path = Path(path)
    if not cfg_path.is_absolute() and not cfg_path.exists():
        cfg_path = _CONFIG_DIR / path
    cfg = OmegaConf.load(cfg_path)
    defaults = cfg.pop("defaults", None)
    if defaults is not None and "data" in defaults:
        data_cfg = OmegaConf.load(_CONFIG_DIR / defaults["data"])
        cfg = OmegaConf.merge({"data": data_cfg}, cfg)
    if overrides:
        cfg = OmegaConf.merge(cfg, OmegaConf.from_dotlist(overrides))
    return cfg

def resolve_device(name: str) -> str:
    if name != "auto":
        return name
    import torch
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"
''')

# ── src/utils/seed.py ────────────────────────────────────────────────────────
with open("src/utils/seed.py", "w") as f:
    f.write('''
from __future__ import annotations
import os, random
import numpy as np
import torch

def set_seed(seed: int, deterministic: bool = True) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
''')

print("Utils written.")

In [ ]:
# ── src/data/kaggle_adapter.py ───────────────────────────────────────────────
with open("src/data/kaggle_adapter.py", "w") as f:
    f.write('''
from __future__ import annotations
import ast
from pathlib import Path
import numpy as np
import pandas as pd
from omegaconf import DictConfig


def load(cfg: DictConfig) -> list[dict]:
    csv_path = Path(cfg.paths.kaggle_csv)
    images_dir = Path(cfg.paths.images)
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    if cfg.subset.require_image:
        df = df[df["Image_Name"].notna() & (df["Image_Name"].str.strip() != "")]
    rng = np.random.default_rng(cfg.subset.seed)
    idx = rng.permutation(len(df))
    df = df.iloc[idx].reset_index(drop=True)
    if cfg.subset.n_recipes > 0:
        df = df.iloc[: cfg.subset.n_recipes]
    recipes = []
    for row_idx, row in df.iterrows():
        image_name = str(row["Image_Name"]).strip()
        recipes.append({
            "id": str(row_idx),
            "title": str(row["Title"]).strip(),
            "ingredients": _parse_ingredients(row["Ingredients"]),
            "instructions": _parse_instructions(row["Instructions"]),
            "image_path": str(images_dir / f"{image_name}.jpg"),
            "partition": "",
        })
    _assign_splits(recipes, cfg.splits)
    return recipes


def _parse_ingredients(raw) -> list[str]:
    if isinstance(raw, list):
        return [str(i).strip() for i in raw if str(i).strip()]
    raw = str(raw).strip()
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(i).strip() for i in parsed if str(i).strip()]
    except (ValueError, SyntaxError):
        pass
    return [s.strip() for s in raw.split(",") if s.strip()]


def _parse_instructions(raw) -> str:
    if isinstance(raw, list):
        return " ".join(str(s).strip() for s in raw)
    raw = str(raw).strip()
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return " ".join(str(s).strip() for s in parsed)
    except (ValueError, SyntaxError):
        pass
    return raw


def _assign_splits(recipes: list[dict], splits_cfg: DictConfig) -> None:
    n = len(recipes)
    n_train = round(n * splits_cfg.train_frac)
    n_val = round(n * splits_cfg.val_frac)
    for i, recipe in enumerate(recipes):
        if i < n_train:
            recipe["partition"] = "train"
        elif i < n_train + n_val:
            recipe["partition"] = "val"
        else:
            recipe["partition"] = "test"
''')

print("Data adapter written.")

In [ ]:
# ── src/data/build_dataset.py ────────────────────────────────────────────────
with open("src/data/build_dataset.py", "w") as f:
    f.write('''
from __future__ import annotations
import logging
from pathlib import Path
import torch
from omegaconf import DictConfig
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from src.data import kaggle_adapter

_log = logging.getLogger(__name__)


class RecipeDataset(Dataset):
    def __init__(self, cfg: DictConfig, partition: str | None = None) -> None:
        recipes = kaggle_adapter.load(cfg.data)
        if partition is not None:
            recipes = [r for r in recipes if r["partition"] == partition]
        feat_path = Path(cfg.data.paths.image_feats)
        if not feat_path.exists():
            raise FileNotFoundError(f"Image features not found at {feat_path}. Run precompute first.")
        feats_dict: dict[str, torch.Tensor] = torch.load(feat_path, weights_only=False)
        before = len(recipes)
        recipes = [r for r in recipes if Path(r["image_path"]).stem in feats_dict]
        dropped = before - len(recipes)
        if dropped:
            _log.warning("Dropped %d recipes missing from image_feats", dropped)
        self.recipes = recipes
        self.feats_dict = feats_dict
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.data.text.tokenizer)
        self.cfg = cfg

    def __len__(self) -> int:
        return len(self.recipes)

    def __getitem__(self, idx: int) -> dict:
        recipe = self.recipes[idx]
        stem = Path(recipe["image_path"]).stem
        image_feat = self.feats_dict[stem].float()
        ingr_enc = self.tokenizer(
            ", ".join(recipe["ingredients"]),
            max_length=self.cfg.data.text.ingr_max_tokens,
            truncation=True, padding="max_length", return_tensors="pt",
        )
        instr_enc = self.tokenizer(
            recipe["instructions"],
            max_length=self.cfg.data.text.instr_max_tokens,
            truncation=True, padding="max_length", return_tensors="pt",
        )
        return {
            "image_feat": image_feat,
            "ingr_input_ids": ingr_enc["input_ids"].squeeze(0),
            "ingr_attention_mask": ingr_enc["attention_mask"].squeeze(0),
            "instr_input_ids": instr_enc["input_ids"].squeeze(0),
            "instr_attention_mask": instr_enc["attention_mask"].squeeze(0),
            "recipe_id": recipe["id"],
            "partition": recipe["partition"],
        }


def get_split(cfg: DictConfig, partition: str) -> RecipeDataset:
    if partition not in {"train", "val", "test"}:
        raise ValueError(f"Unknown partition {partition!r}")
    return RecipeDataset(cfg, partition=partition)
''')

print("Dataset written.")

In [ ]:
# ── src/models/*.py ───────────────────────────────────────────────────────────
with open("src/models/image_encoder.py", "w") as f:
    f.write('''
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F

class ImageEncoder(nn.Module):
    def __init__(self, in_dim: int, proj_hidden: int, embed_dim: int) -> None:
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, proj_hidden),
            nn.LayerNorm(proj_hidden),
            nn.GELU(),
            nn.Linear(proj_hidden, embed_dim),
            nn.LayerNorm(embed_dim),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.proj(x), dim=-1)
''')

with open("src/models/text_encoder.py", "w") as f:
    f.write('''
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class TextEncoder(nn.Module):
    def __init__(self, encoder_name: str, hidden: int, embed_dim: int, freeze: bool = False) -> None:
        super().__init__()
        self.bert = AutoModel.from_pretrained(encoder_name)
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad_(False)
        self.ingr_proj = nn.Sequential(nn.Linear(hidden, embed_dim), nn.LayerNorm(embed_dim))
        self.instr_proj = nn.Sequential(nn.Linear(hidden, embed_dim), nn.LayerNorm(embed_dim))

    @staticmethod
    def _masked_mean(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask_f = mask.unsqueeze(-1).float()
        return (hidden * mask_f).sum(1) / mask_f.sum(1).clamp(min=1e-9)

    def encode_ingr(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._masked_mean(out.last_hidden_state, attention_mask)
        emb = F.normalize(self.ingr_proj(pooled), dim=-1)
        return emb, out.last_hidden_state

    def encode_instr(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._masked_mean(out.last_hidden_state, attention_mask)
        return F.normalize(self.instr_proj(pooled), dim=-1)
''')

with open("src/models/fusion.py", "w") as f:
    f.write('''
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F

_MODES = {"concat", "attention", "ingr_only"}

class FusionModule(nn.Module):
    def __init__(self, embed_dim: int, hidden: int, mode: str, n_heads: int = 8, text_hidden: int = 768) -> None:
        super().__init__()
        if mode not in _MODES:
            raise ValueError(f"Unknown fusion mode {mode!r}")
        self.mode = mode
        if mode == "concat":
            self.mlp = nn.Sequential(
                nn.Linear(embed_dim * 2, hidden),
                nn.GELU(),
                nn.Linear(hidden, embed_dim),
                nn.LayerNorm(embed_dim),
            )
        elif mode == "attention":
            self.ingr_token_proj = nn.Linear(text_hidden, embed_dim)
            self.cross_attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
            self.norm = nn.LayerNorm(embed_dim)

    def forward(self, ingr_emb, instr_emb, ingr_hidden=None, ingr_mask=None):
        if self.mode == "concat":
            x = torch.cat([ingr_emb, instr_emb], dim=-1)
            return F.normalize(self.mlp(x), dim=-1)
        if self.mode == "attention":
            kv = self.ingr_token_proj(ingr_hidden)
            q = instr_emb.unsqueeze(1)
            key_padding_mask = ingr_mask == 0
            attended, _ = self.cross_attn(q, kv, kv, key_padding_mask=key_padding_mask)
            out = self.norm(attended.squeeze(1) + instr_emb)
            return F.normalize(out, dim=-1)
        return F.normalize(ingr_emb, dim=-1)
''')

with open("src/models/joint_embedding.py", "w") as f:
    f.write('''
from __future__ import annotations
import torch
import torch.nn as nn
from omegaconf import DictConfig
from src.models.image_encoder import ImageEncoder
from src.models.text_encoder import TextEncoder
from src.models.fusion import FusionModule

class JointEmbeddingModel(nn.Module):
    def __init__(self, cfg: DictConfig) -> None:
        super().__init__()
        m = cfg.model
        self.image_encoder = ImageEncoder(m.image.in_dim, m.image.proj_hidden, m.embed_dim)
        self.text_encoder = TextEncoder(m.text.encoder, m.text.hidden, m.embed_dim, m.text.freeze_text)
        self.fusion = FusionModule(
            embed_dim=m.embed_dim, hidden=m.fusion.hidden, mode=m.fusion.mode,
            n_heads=m.fusion.get("n_heads", 8), text_hidden=m.text.hidden,
        )

    def forward(self, batch: dict):
        image_emb = self.image_encoder(batch["image_feat"])
        ingr_emb, ingr_hidden = self.text_encoder.encode_ingr(
            batch["ingr_input_ids"], batch["ingr_attention_mask"]
        )
        instr_emb = self.text_encoder.encode_instr(
            batch["instr_input_ids"], batch["instr_attention_mask"]
        )
        recipe_emb = self.fusion(
            ingr_emb, instr_emb,
            ingr_hidden=ingr_hidden, ingr_mask=batch["ingr_attention_mask"],
        )
        return image_emb, recipe_emb
''')

print("Models written.")

In [ ]:
# ── src/losses/infonce.py ─────────────────────────────────────────────────────
with open("src/losses/infonce.py", "w") as f:
    f.write('''
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F

class InfoNCELoss(nn.Module):
    def __init__(self, temperature: float = 0.07) -> None:
        super().__init__()
        self.temperature = temperature

    def forward(self, image_emb: torch.Tensor, recipe_emb: torch.Tensor) -> torch.Tensor:
        B = image_emb.size(0)
        labels = torch.arange(B, device=image_emb.device)
        logits_i2r = image_emb @ recipe_emb.T / self.temperature
        logits_r2i = recipe_emb @ image_emb.T / self.temperature
        return 0.5 * (F.cross_entropy(logits_i2r, labels) + F.cross_entropy(logits_r2i, labels))
''')

# ── src/eval/metrics.py ───────────────────────────────────────────────────────
with open("src/eval/metrics.py", "w") as f:
    f.write('''
from __future__ import annotations
import torch

def compute_metrics(
    image_embs: torch.Tensor,
    recipe_embs: torch.Tensor,
    ks: tuple[int, ...] = (1, 5, 10),
) -> dict[str, float]:
    sim = image_embs @ recipe_embs.T
    results: dict[str, float] = {}
    for direction, s in [("im2recipe", sim), ("recipe2im", sim.T)]:
        diag = s.diagonal().unsqueeze(1)
        ranks = (s > diag).sum(dim=1).float() + 1
        results[f"{direction}_medR"] = ranks.median().item()
        for k in ks:
            results[f"{direction}_R@{k}"] = (ranks <= k).float().mean().item() * 100.0
    return results
''')

print("Loss + eval written.")

## 3. Write configs (Kaggle paths)

In [ ]:
import os

# Detect Kaggle dataset input path
KAGGLE_INPUT = "/kaggle/input/food-ingredients-and-recipe-dataset-with-images"
CSV_PATH = f"{KAGGLE_INPUT}/Food Ingredients and Recipe Dataset with Image Name Mapping.csv"
IMG_DIR   = f"{KAGGLE_INPUT}/Food Images/Food Images"

assert os.path.exists(CSV_PATH), f"CSV not found: {CSV_PATH}"
assert os.path.isdir(IMG_DIR),   f"Image dir not found: {IMG_DIR}"

n_images = len(list(__import__('pathlib').Path(IMG_DIR).glob('*.jpg')))
print(f"Dataset: {CSV_PATH}")
print(f"Images : {IMG_DIR}  ({n_images} .jpg files)")

In [ ]:
DATA_YAML = f"""
paths:
  raw: /kaggle/input/food-ingredients-and-recipe-dataset-with-images
  processed: /kaggle/working/data/processed
  kaggle_csv: "{CSV_PATH}"
  images: "{IMG_DIR}"
  image_feats: /kaggle/working/data/processed/image_feats.pt

subset:
  n_recipes: 0           # 0 = use all available recipes
  require_image: true
  seed: 42

splits:
  train_frac: 0.80
  val_frac: 0.10
  test_frac: 0.10

text:
  ingr_max_tokens: 24
  instr_max_tokens: 128
  tokenizer: distilbert-base-uncased

image:
  clip_model: ViT-B-32
  clip_pretrained: openai
  feat_dim: 512

semantic:
  enabled: true
  n_categories: 20
"""

with open("configs/data.yaml", "w") as f:
    f.write(DATA_YAML)

BASELINE_YAML = """
defaults:
  data: data.yaml

exp_name: baseline_concat
seed: 42
device: auto

model:
  embed_dim: 1024
  image:
    in_dim: 512
    proj_hidden: 1024
  text:
    encoder: distilbert-base-uncased
    hidden: 768
    freeze_text: false
    share_encoder: true
  fusion:
    mode: concat
    hidden: 1024
    pool: mean
  semantic:
    enabled: true
    n_categories: 20

loss:
  name: infonce
  temperature: 0.07
  lambda_sem: 0.1
  triplet_margin: 0.3

train:
  epochs: 30
  batch_size: 128
  lr: 1.0e-4
  weight_decay: 0.01
  grad_accum: 1
  amp: true
  early_stop_patience: 5
  num_workers: 2

eval:
  subsets: [1k]
  directions: both
  n_folds: 10
  seed: 1234

log:
  backend: tensorboard
  dir: /kaggle/working/runs
"""

FUSION_YAML = """
defaults:
  data: data.yaml

exp_name: fusion_attention
seed: 42
device: auto

model:
  embed_dim: 1024
  image:
    in_dim: 512
    proj_hidden: 1024
  text:
    encoder: distilbert-base-uncased
    hidden: 768
    freeze_text: false
    share_encoder: true
  fusion:
    mode: attention
    hidden: 1024
    n_heads: 8
    pool: attention
  semantic:
    enabled: true
    n_categories: 20

loss:
  name: infonce
  temperature: 0.07
  lambda_sem: 0.1
  triplet_margin: 0.3

train:
  epochs: 30
  batch_size: 128
  lr: 1.0e-4
  weight_decay: 0.01
  grad_accum: 1
  amp: true
  early_stop_patience: 5
  num_workers: 2

eval:
  subsets: [1k]
  directions: both
  n_folds: 10
  seed: 1234

log:
  backend: tensorboard
  dir: /kaggle/working/runs
"""

with open("configs/baseline.yaml", "w") as f:
    f.write(BASELINE_YAML)
with open("configs/fusion.yaml", "w") as f:
    f.write(FUSION_YAML)

print("Configs written.")

## 4. Precompute CLIP ViT-B/32 image features

Runs once. Saves `{stem: Tensor(512)}` dict to `data/processed/image_feats.pt`.

In [ ]:
import logging
import sys
from pathlib import Path

import open_clip
import torch
from PIL import Image
from tqdm.notebook import tqdm

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(levelname)s — %(message)s")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEAT_PATH = Path("/kaggle/working/data/processed/image_feats.pt")
FEAT_PATH.parent.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 256

print(f"Device: {DEVICE}")

if FEAT_PATH.exists():
    existing = torch.load(FEAT_PATH, weights_only=False)
    print(f"Features already exist: {len(existing)} images. Skipping precompute.")
else:
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai", device=DEVICE
    )
    model.eval()

    jpg_paths = sorted(Path(IMG_DIR).glob("*.jpg"))
    print(f"Encoding {len(jpg_paths)} images...")

    feats_dict = {}
    skipped = 0

    for i in tqdm(range(0, len(jpg_paths), BATCH_SIZE), desc="CLIP encode"):
        batch_paths = jpg_paths[i : i + BATCH_SIZE]
        imgs, stems = [], []
        for p in batch_paths:
            try:
                imgs.append(preprocess(Image.open(p).convert("RGB")))
                stems.append(p.stem)
            except Exception as e:
                logging.warning("Skip %s: %s", p.name, e)
                skipped += 1
        if not imgs:
            continue
        batch = torch.stack(imgs).to(DEVICE)
        with torch.no_grad():
            feats = model.encode_image(batch)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        for stem, feat in zip(stems, feats.cpu()):
            feats_dict[stem] = feat

    torch.save(feats_dict, FEAT_PATH)
    print(f"Saved {len(feats_dict)} features to {FEAT_PATH}  (skipped {skipped})")

## 5. Training helpers

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

import logging
import torch
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader

from src.data.build_dataset import get_split
from src.eval.metrics import compute_metrics
from src.losses.infonce import InfoNCELoss
from src.models.joint_embedding import JointEmbeddingModel
from src.utils.config import load_config, resolve_device
from src.utils.seed import set_seed

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s — %(message)s")


def _move(batch, device):
    return {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}


@torch.no_grad()
def _embed_split(model, loader, device):
    model.eval()
    imgs, recs = [], []
    for batch in loader:
        batch = _move(batch, device)
        ie, re = model(batch)
        imgs.append(ie.cpu())
        recs.append(re.cpu())
    return torch.cat(imgs), torch.cat(recs)


def _eval_kfold(image_embs, recipe_embs, k, n_folds, seed):
    N = image_embs.size(0)
    k = min(k, N)
    fold_metrics = []
    for fold in range(n_folds):
        g = torch.Generator()
        g.manual_seed(seed + fold)
        idx = torch.randperm(N, generator=g)[:k]
        fold_metrics.append(compute_metrics(image_embs[idx], recipe_embs[idx]))
    avg = {}
    for key in fold_metrics[0]:
        avg[key] = sum(m[key] for m in fold_metrics) / n_folds
    return avg


def train_one_run(config_name: str, seed: int, epochs: int | None = None) -> dict:
    """Train model and return best val + test metrics."""
    cfg = load_config(config_name, overrides=[f"seed={seed}", f"exp_name={config_name.replace('.yaml','')}_{seed}"])
    if epochs is not None:
        cfg.train.epochs = epochs

    set_seed(cfg.seed)
    device = resolve_device(cfg.device)
    amp_enabled = cfg.train.amp and device == "cuda"
    autocast_dev = "cuda" if device == "cuda" else "cpu"

    train_ds = get_split(cfg, "train")
    val_ds   = get_split(cfg, "val")
    test_ds  = get_split(cfg, "test")

    pin = device == "cuda"
    kw = dict(batch_size=cfg.train.batch_size, num_workers=cfg.train.num_workers, pin_memory=pin)
    train_loader = DataLoader(train_ds, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)

    model = JointEmbeddingModel(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[seed={seed}] model params: {n_params/1e6:.1f}M  train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

    criterion = InfoNCELoss(temperature=cfg.loss.temperature)
    optimizer = AdamW(model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
    scaler = GradScaler(enabled=amp_enabled)

    best_medR = float("inf")
    patience = 0
    best_state = None

    for epoch in range(cfg.train.epochs):
        model.train()
        optimizer.zero_grad()
        epoch_loss = 0.0
        for step, batch in enumerate(train_loader):
            batch = _move(batch, device)
            with autocast(autocast_dev, enabled=amp_enabled):
                ie, re = model(batch)
                loss = criterion(ie, re) / cfg.train.grad_accum
            scaler.scale(loss).backward()
            if (step + 1) % cfg.train.grad_accum == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            epoch_loss += loss.item() * cfg.train.grad_accum

        # val eval (1k fold)
        ie, re = _embed_split(model, val_loader, device)
        val_m = _eval_kfold(ie, re, k=1000, n_folds=cfg.eval.n_folds, seed=cfg.eval.seed)
        val_medR = val_m["im2recipe_medR"]
        print(
            f"  epoch {epoch+1:02d}  loss={epoch_loss/len(train_loader):.4f}  "
            f"val medR={val_medR:.1f}  R@1={val_m['im2recipe_R@1']:.1f}  R@10={val_m['im2recipe_R@10']:.1f}"
        )

        if val_medR < best_medR:
            best_medR = val_medR
            patience = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= cfg.train.early_stop_patience:
                print(f"  Early stop at epoch {epoch+1}")
                break

    # restore best and eval test
    if best_state is not None:
        model.load_state_dict(best_state)

    test_ie, test_re = _embed_split(model, test_loader, device)

    # 1k eval (10-fold sampling)
    m1k = _eval_kfold(test_ie, test_re, k=1000, n_folds=cfg.eval.n_folds, seed=cfg.eval.seed)

    # full-gallery eval (all test samples as gallery)
    m_full = compute_metrics(test_ie, test_re)

    print(f"  [seed={seed}] TEST 1k  im2recipe medR={m1k['im2recipe_medR']:.1f}  R@1={m1k['im2recipe_R@1']:.1f}  R@10={m1k['im2recipe_R@10']:.1f}")
    print(f"  [seed={seed}] TEST full im2recipe medR={m_full['im2recipe_medR']:.1f}  R@1={m_full['im2recipe_R@1']:.1f}  R@10={m_full['im2recipe_R@10']:.1f}")

    return {"1k": m1k, "full": m_full, "best_val_medR": best_medR}


print("Training helpers loaded.")

## 6. Train Baseline (concat fusion) × 3 seeds

In [ ]:
SEEDS = [42, 123, 777]

print("=" * 60)
print("BASELINE — concat fusion")
print("=" * 60)

baseline_results = []
for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    r = train_one_run("baseline.yaml", seed=seed)
    baseline_results.append(r)

print("\nBaseline runs complete.")

## 7. Train Attention Fusion × 3 seeds

In [ ]:
print("=" * 60)
print("FUSION — cross-attention")
print("=" * 60)

fusion_results = []
for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    r = train_one_run("fusion.yaml", seed=seed)
    fusion_results.append(r)

print("\nFusion runs complete.")

## 8. Results table — mean ± std over 3 seeds

In [ ]:
import numpy as np

METRICS = [
    "im2recipe_medR", "im2recipe_R@1", "im2recipe_R@5", "im2recipe_R@10",
    "recipe2im_medR", "recipe2im_R@1", "recipe2im_R@5", "recipe2im_R@10",
]

def summarize(results, subset_key):
    out = {}
    for m in METRICS:
        vals = [r[subset_key][m] for r in results]
        out[m] = (np.mean(vals), np.std(vals))
    return out

b1k   = summarize(baseline_results, "1k")
bfull = summarize(baseline_results, "full")
f1k   = summarize(fusion_results,   "1k")
ffull = summarize(fusion_results,   "full")

def row(name, d):
    vals = [f"{d[m][0]:.1f}±{d[m][1]:.1f}" for m in METRICS]
    return f"| {name:<22} | " + " | ".join(f"{v:>12}" for v in vals) + " |"

header  = "| Model                  | " + " | ".join(f"{m:>12}" for m in METRICS) + " |"
divider = "|" + "-" * 24 + "|" + ("|" + "-" * 14) * len(METRICS) + "|"

print("\n## Retrieval Results (mean ± std, 3 seeds)")
print("\n### 1k Gallery (10-fold)")
print(header)
print(divider)
print(row("Baseline (concat) 1k", b1k))
print(row("Fusion (attention) 1k", f1k))

print("\n### Full Test Gallery")
print(header)
print(divider)
print(row("Baseline (concat) full", bfull))
print(row("Fusion (attention) full", ffull))

In [ ]:
# Per-seed breakdown
print("\n## Per-seed detail — Baseline")
for seed, r in zip(SEEDS, baseline_results):
    m = r["1k"]
    print(f"  seed={seed}  im2r medR={m['im2recipe_medR']:.1f}  R@1={m['im2recipe_R@1']:.1f}  "
          f"R@10={m['im2recipe_R@10']:.1f}  r2im medR={m['recipe2im_medR']:.1f}  R@1={m['recipe2im_R@1']:.1f}")

print("\n## Per-seed detail — Fusion")
for seed, r in zip(SEEDS, fusion_results):
    m = r["1k"]
    print(f"  seed={seed}  im2r medR={m['im2recipe_medR']:.1f}  R@1={m['im2recipe_R@1']:.1f}  "
          f"R@10={m['im2recipe_R@10']:.1f}  r2im medR={m['recipe2im_medR']:.1f}  R@1={m['recipe2im_R@1']:.1f}")

In [ ]:
# Save results to disk
import json

output = {
    "seeds": SEEDS,
    "baseline": baseline_results,
    "fusion": fusion_results,
}

# Convert numpy floats to Python floats for JSON serialization
def _to_python(obj):
    if isinstance(obj, dict):
        return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_python(v) for v in obj]
    if hasattr(obj, 'item'):
        return obj.item()
    return obj

with open("/kaggle/working/phase4_results.json", "w") as f:
    json.dump(_to_python(output), f, indent=2)

print("Results saved to /kaggle/working/phase4_results.json")